# CodeAlpha Task 1: Web Scraping | Masoko Kenya Electronics

**Author:** Evans Kiplangat | **Program:** CodeAlpha Data Analytics Internship

This notebook builds a clean, analysis-ready dataset of Kenyan consumer-electronics listings.
It covers source vetting, ethical scraping, extraction, cleaning, feature engineering and validation.
The output feeds Task 2 (EDA), Task 3 (Visualization) and the product-side analysis of the project.

**Pipeline:** vet sources -> check robots.txt -> scrape -> clean -> engineer features -> validate -> save

| Section | What happens |
|---|---|
| 1 | Setup and configuration |
| 2 | robots.txt compliance check |
| 3 | Scraper (pagination, retries, polite delays) |
| 4 | Cleaning and feature engineering |
| 5 | Validation and export |

## 1. Setup and configuration

**Source selection (done before this notebook):** nine Kenyan sites were tested.
Jumia sits behind a Cloudflare challenge, Kilimall's robots.txt disallows `/product*`, `/item*` and `/api/*`,
and Jiji, Carrefour, Naivas and PigiaMe returned 403. Masoko's robots.txt allows all crawling, so it was selected.

In [ ]:
import json, time, random, re, os, shutil
from datetime import datetime
from urllib import robotparser

import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.masoko.com"
HEADERS = {"User-Agent": "Mozilla/5.0 (educational project; CodeAlpha internship)"}

# Category paths scraped. 'computing/routers' returned no data (different page structure) and is excluded.
CATEGORIES = [
    "phones-accessories/mobile-phones",
    "phones-accessories/accessories",
    "laptops",
    "apple-devices",
    "gaming-items",
]

DELAY_RANGE = (1.5, 2.5)          # seconds between requests
DATA_DIR = "data"
RAW_CSV = f"{DATA_DIR}/masoko_products_raw.csv"
CLEAN_CSV = f"{DATA_DIR}/masoko_products_clean.csv"
os.makedirs(DATA_DIR, exist_ok=True)

## 2. robots.txt compliance check

The scraper checks every URL against the site's robots.txt before requesting it.

In [ ]:
r = requests.get(f"{BASE}/robots.txt", headers=HEADERS, timeout=15)
print("Status:", r.status_code)
print(r.text[:500])

rp = robotparser.RobotFileParser()
rp.parse(r.text.splitlines())

for cat in CATEGORIES:
    allowed = rp.can_fetch(HEADERS["User-Agent"], f"{BASE}/{cat}")
    print(f"{'ALLOWED' if allowed else 'BLOCKED':8} /{cat}")

## 3. Scraper

Masoko is a Next.js site: each category page embeds its product list as JSON inside a
`<script id="__NEXT_DATA__">` tag. Parsing that JSON is more reliable than scraping HTML tags,
and it means each request returns structured fields (price, special price, stock, SKU).

Design choices:
- one `requests.Session` reused across calls
- random 1.5-2.5 s delay between pages
- up to 3 retries with increasing back-off on timeouts or errors
- pagination follows the site's own `nextPage` value until it is empty

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)

def fetch_category_page(path, page, retries=3):
    """Return the categoryPage dict for one listing page, or None if it cannot be fetched."""
    url = f"{BASE}/{path}" + (f"?page={page}" if page > 1 else "")
    if not rp.can_fetch(HEADERS["User-Agent"], url):
        print(f"  skipped (robots.txt): {url}")
        return None
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, timeout=20)
            if r.status_code == 200:
                tag = BeautifulSoup(r.text, "html.parser").find("script", id="__NEXT_DATA__")
                if tag:
                    return json.loads(tag.string)["props"]["pageProps"]["category"]["categoryPage"]
            else:
                print(f"  {url} -> HTTP {r.status_code} (attempt {attempt})")
        except (requests.RequestException, KeyError, TypeError, json.JSONDecodeError) as e:
            print(f"  {type(e).__name__} (attempt {attempt})")
        time.sleep(3 * attempt)
    return None

In [ ]:
all_rows = []
for cat in CATEGORIES:
    print(f"\nScraping: {cat}")
    page = 1
    while page:
        cp = fetch_category_page(cat, page)
        if not cp or not cp.get("items"):
            print(f"  page {page}: no data, moving on")
            break
        for item in cp["items"]:
            item["category"] = cat
            item["page"] = page
            item["scraped_at"] = datetime.now().isoformat(timespec="seconds")
            all_rows.append(item)
        pag = cp.get("pagination", {})
        total_pages = -(-pag.get("totalCount", 0) // pag.get("pageSize", 20))
        print(f"  page {page}/{total_pages} | rows so far: {len(all_rows)}")
        page = pag.get("nextPage")
        time.sleep(random.uniform(*DELAY_RANGE))

raw = pd.json_normalize(all_rows)
before = len(raw)
raw = raw.drop_duplicates(subset="id", keep="first")   # the same product can sit in several categories
print(f"\nRaw rows: {before} | unique products: {len(raw)}")

# Nested lists/dicts -> JSON strings so the CSV round-trips cleanly
for col in raw.columns:
    if raw[col].apply(lambda x: isinstance(x, (list, dict))).any():
        raw[col] = raw[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)

raw.to_csv(RAW_CSV, index=False)
print("Saved", RAW_CSV, raw.shape)
print(raw["category"].value_counts())

## 4. Cleaning and feature engineering

Findings from inspecting the raw data that drive the steps below:
- `price` and `specialPrice` arrive as text
- rating, likes, vendor and several other fields exist in the site's schema but are **empty for every row**, so they are dropped
- brand and specs (RAM, storage, screen size) are only available inside the product name, so they are extracted with regex
- gaming items mix hardware, video games and gift cards, so they need a product type label

In [ ]:
df = pd.read_csv(RAW_CSV)

# 1. Drop fields that are empty for every row, plus the near-empty insurance column
EMPTY_COLS = ["maxPrice", "likes", "vendorName", "rating", "deviceFinancing",
              "deviceInsurancePercent", "configurableOptions", "__typename",
              "stock.__typename", "deviceInsuranceMode"]
df = df.drop(columns=EMPTY_COLS, errors="ignore")
df = df.rename(columns={"stock.isInStock": "in_stock", "stock.qty": "stock_qty"})

# in_stock is constant (every listing is in stock), so it carries no information
if df["in_stock"].nunique() == 1:
    df = df.drop(columns=["in_stock"])

# 2. Numeric prices and discount features
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["specialPrice"] = pd.to_numeric(df["specialPrice"], errors="coerce")
df["price_anomaly"] = (df["price"] <= 0) | (df["specialPrice"] >= df["price"])
df["has_discount"] = df["specialPrice"].notna() & (df["specialPrice"] < df["price"])
df["final_price"] = np.where(df["has_discount"], df["specialPrice"], df["price"])
df["discount_kes"] = np.where(df["has_discount"], df["price"] - df["specialPrice"], 0)
df["discount_pct"] = (df["discount_kes"] / df["price"] * 100).round(2)
print("Prices and discounts done:", df.shape)

In [ ]:
# 3. Brand from the product name
APPLE_WORDS = {"iphone", "ipad", "macbook", "airpods", "apple", "imac", "ipod"}
SHORT_BRANDS = {"Jbl": "JBL", "Hp": "HP", "Lg": "LG", "Jvc": "JVC", "Tcl": "TCL",
                "Msi": "MSI", "Zte": "ZTE", "Oppo": "OPPO", "Oneplus": "OnePlus"}

def get_brand(name):
    first = re.split(r"[\s:\-]+", str(name).strip())[0].lower()
    return "Apple" if first in APPLE_WORDS else first.title()

def format_brand(b):
    if b in SHORT_BRANDS:
        return SHORT_BRANDS[b]
    return b.upper() if len(b) <= 3 else b

df["brand"] = df["name"].apply(get_brand).apply(format_brand)

# 4. Specs from the product name
def first_num(pattern, s):
    m = re.search(pattern, str(s), flags=re.I)
    return float(m.group(1)) if m else np.nan

def storage_gb(s):
    m = re.search(r"(\d+)\s*(GB|TB)\s*(?:ROM|SSD|HDD|STORAGE)", str(s), flags=re.I)
    if not m:
        return np.nan
    v = float(m.group(1))
    return v * 1024 if m.group(2).upper() == "TB" else v

df["ram_gb"] = df["name"].apply(lambda s: first_num(r"(\d+)\s*GB\s*RAM", s))
df["storage_gb"] = df["name"].apply(storage_gb)
df["screen_in"] = df["name"].apply(lambda s: first_num(r"(\d+(?:\.\d+)?)\s*(?:''|\"|\u201d|inch)", s))

# 5. Category helpers
df["category_group"] = df["category"].str.split("/").str[-1]
df["has_campaign"] = df["campaign"].fillna("[]").astype(str).ne("[]")
print("Brand and specs done:", df.shape)

In [ ]:
# 6. Product type. Gaming items mix hardware, video games and gift cards, so the first-word
#    brand rule produces game titles as 'brands'. Classify them explicitly instead.
GAMING_HARDWARE = {"Sony", "Playstation", "Nintendo", "Xbox", "Microsoft", "Logitech", "Razer",
                   "Steelseries", "Corsair", "Hyperx", "Havit", "JBL", "Canyon", "Redragon",
                   "Hama", "Thrustmaster", "Meta", "Oculus"}

def is_gift_card(name):
    n = str(name).strip()
    return n.startswith("$") or "gift card" in n.lower() or "wallet" in n.lower()

def classify(row):
    if row["category_group"] != "gaming-items":
        return row["brand"], "Hardware/Accessory"
    if is_gift_card(row["name"]):
        return "Gift Card", "Gift Card"
    if row["brand"] in GAMING_HARDWARE:
        return row["brand"], "Hardware/Accessory"
    return "Video Game", "Video Game"

res = df.apply(classify, axis=1, result_type="expand")
df["brand"], df["product_type"] = res[0], res[1]

# 7. Manual brand corrections found during inspection
df.loc[df["brand"].isin({"Homepod", "MAC"}), "brand"] = "Apple"            # HomePod, Mac Mini
df.loc[df["name"].str.startswith("Vision Plus"), "brand"] = "Vision Plus"  # two-word brand

# Single-occurrence 'brands' that are really product lines (stands, UPS units) -> Other
LEGIT_SINGLETONS = {"Google", "Dell", "Lenovo", "Amazon", "Neon", "Yeelight"}
vc = df["brand"].value_counts()
rare = set(vc[vc == 1].index) - LEGIT_SINGLETONS
df.loc[df["brand"].isin(rare), "brand"] = "Other"

# 8. SKU structure: first 3 letters = product type code, remaining letters = source/channel code
sku_letters = df["sku"].astype(str).str.extract(r"^([A-Za-z]+)")[0]
df["sku_type"] = sku_letters.str[:3].str.upper()
df["sku_channel"] = sku_letters.str[3:]

print("Cleaning complete:", df.shape)

## 5. Validation and export

In [ ]:
checks = {
    "ids are unique": bool(df["id"].is_unique),
    "no missing final_price": bool(df["final_price"].notna().all()),
    "all list prices positive": bool((df["price"] > 0).all()),
    "special price never above list price": not bool(df["price_anomaly"].any()),
    "no missing brand / product_type / category": bool(df[["brand", "product_type", "category_group"]].notna().all().all()),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)

print("\nRows:", len(df), "| Columns:", df.shape[1], "| Brands:", df["brand"].nunique())
print(f"Discounted listings: {df['has_discount'].mean():.1%}")
print(f"Median final price: KES {df['final_price'].median():,.0f}")
print("\nListings per category:\n", df["category_group"].value_counts().to_string())
print("\nSpec coverage by category (share of rows with a value):")
print(df.groupby("category_group")[["ram_gb", "storage_gb", "screen_in"]].apply(lambda g: g.notna().mean().round(2)))

In [ ]:
df.to_csv(CLEAN_CSV, index=False)
print("Saved", CLEAN_CSV, df.shape)

# Optional: keep a copy on Google Drive when running in Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/CodeAlpha", exist_ok=True)
    for f in (RAW_CSV, CLEAN_CSV):
        shutil.copy(f, "/content/drive/MyDrive/CodeAlpha/")
    print("Copied to Google Drive")
except ImportError:
    pass

## Limitations

- Rating, review, vendor and brand fields exist in the site's data but were empty, so no review-based analysis is possible from this source
- `stock_qty` looks capped (about 41% of rows show exactly 10); treat it as indicative only
- A product listed in several categories is kept under the first category scraped, so smaller categories such as Apple devices are under-counted
- Brand and spec extraction is rule-based on product names and can contain errors
- 37 product names appear twice with different SKU channel codes and a median price gap of about 3%; the cause is not documented by the site
- The data is a single snapshot taken on one date

## Next

Task 2 (EDA) and Task 3 (Visualization) use `data/masoko_products_clean.csv`.